# ESM25 Earthquake Catalogue — Magnitude Distribution Analysis

**Authors:** Spina Cianetti  
**License:** [GPL-3.0](https://www.gnu.org/licenses/gpl-3.0.html) 
This code is released under the GNU General Public License v3.0.

**Repository:** European Strong-Motion Database (ESM-DB)

---

This notebook analyses the **magnitude distribution** of seismic events in the ESM25 catalogue.  
Starting from the per-event flatfile (`events_2_dataframe.csv`), it:

1. **Loads** the event catalogue and deduplicates records by `esm_event_id`
2. **Assigns a preferred magnitude** to each event following a priority order:
   `emec_mw` → `mw` → `ml` → `ms` → `mb` → `md` → `m`
3. **Filters** events to the Europe/Mediterranean region
   (latitude 0–90 °N, longitude −30–60 °E)
4. **Produces diagnostic figures:**
   - Magnitude histogram (linear and log-frequency)
   - Gutenberg–Richter cumulative frequency curve
   - Stacked histogram by magnitude type
   - Two-panel summary figure (histogram + G–R curve)
   - Bar chart of magnitude-type prevalence
   - Focal depth histogram and magnitude-vs-depth scatter plots
   - Magnitude vs. event time scatter plots
   - Violin plots of magnitude distributions by type

### Output files (relative to the configured `OUTPUT_DIR`)
| File | Description |
|------|-------------|
| `nan_magnitude_events.csv` | Events with no valid magnitude in any column |
| `magnitude_histogram.png` | Linear-scale magnitude histogram |
| `magnitude_histogram_log.png` | Log-scale magnitude histogram |
| `gutenberg_richter.png` | Gutenberg–Richter cumulative curve |
| `magnitude_histogram_stacked.png` | Stacked histogram by magnitude type |
| `magnitude_analysis_subplots.png` | Two-panel: histogram + G–R curve |
| `magnitude_type.png` | Bar chart of magnitude-type prevalence |
| `depth_histogram.png` | Focal depth histogram |
| `magnitude_vs_depth.png` | Magnitude vs. depth scatter (all types) |
| `magnitude_vs_depth_subplots.png` | Per-type magnitude vs. depth subplots |
| `magnitude_vs_depth_subplots_combined_colored.png` | Combined coloured subplots (md+m merged) |
| `magnitude_vs_time_subplots.png` | Per-type magnitude vs. time subplots |
| `magnitude_vs_time_subplots_combined_colored.png` | Combined coloured subplots (md+m merged) |
| `magnitude_violinplot.png` | Violin plots of magnitude by type |

---


## 1. Configuration

In [ ]:
# ── Google Drive mount (remove if running locally) ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Root directory shared among collaborators
SHARED_ROOT = '/content/drive/Shareddrives/ESM25_ORFEUS_Privato/Processamento/'

# Input catalogue CSV
INPUT_CSV = os.path.join(SHARED_ROOT, 'events_dataframe.csv')

# Directory where all output figures and CSVs will be saved
OUTPUT_DIR = SHARED_ROOT

# Priority order for selecting the preferred magnitude type per event
MAGNITUDE_COLS = ['emec_mw', 'mw', 'ml', 'ms', 'mb', 'md', 'm']

# Geographic bounding box for event filtering (Europe / Mediterranean)
LAT_MIN, LAT_MAX   =   0.0,  90.0   # degrees North
LON_MIN, LON_MAX   = -30.0,  60.0   # degrees East


## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns


## 3. Load the event catalogue

Columns that may contain mixed types are explicitly cast to `str` to avoid
pandas inference warnings.

In [ ]:
# Columns known to contain mixed types — read as strings to avoid inference warnings
MIXED_TYPE_COLS = {
    18: str, 22: str, 24: str, 26: str, 27: str, 28: str,
    33: str, 36: str, 38: str, 41: str, 58: str, 75: str,
}

df = pd.read_csv(INPUT_CSV, dtype=MIXED_TYPE_COLS)
print(f"Catalogue loaded: {df.shape[0]:,} records, {df.shape[1]} columns")
df.head(3)


## 4. Deduplicate by event ID

Each `esm_event_id` may appear in multiple rows (one per recording station).
We aggregate to a single row per event, taking the first non-null value in
every column.

In [ ]:
agg_dict = {col: 'first' for col in df.columns}
df_events = df.groupby('esm_event_id').agg(agg_dict).reset_index(drop=True)
print(f"Unique events: {len(df_events):,}")
df_events.head(3)


## 5. Assign preferred magnitude

For each event, the preferred magnitude is the first non-NaN value found
in the priority list `MAGNITUDE_COLS`.  
A companion column `magnitude_type` records which scale was used.

In [ ]:
def get_preferred_magnitude(row):
    """Return the preferred magnitude value and its scale name.

    Iterates through MAGNITUDE_COLS in priority order and returns the first
    non-NaN value together with the column name (scale identifier).

    Returns:
        pd.Series: [magnitude_value, magnitude_type]
    """
    for mag_type in MAGNITUDE_COLS:
        if pd.notna(row[mag_type]):
            return pd.Series([row[mag_type], mag_type])
    return pd.Series([np.nan, np.nan])

df_events[['magnitude', 'magnitude_type']] = df_events.apply(
    get_preferred_magnitude, axis=1
)

n_nan = df_events['magnitude'].isna().sum()
print(f"Events with a valid preferred magnitude : {len(df_events) - n_nan:,}")
print(f"Events with NO valid magnitude          : {n_nan:,}")

df_events[['magnitude', 'magnitude_type'] + MAGNITUDE_COLS].head(5)


## 6. Export events with no valid magnitude

Events for which none of the magnitude columns contain a numeric value are
saved to a CSV for manual inspection.

In [ ]:
nan_mag_path = os.path.join(OUTPUT_DIR, 'nan_magnitude_events.csv')

df_nan = df_events[df_events['magnitude'].isna()][
    ['esm_event_id'] + MAGNITUDE_COLS
]
df_nan.to_csv(nan_mag_path, index=False)
print(f"Saved {len(df_nan):,} events with NaN magnitude → {nan_mag_path}")
df_nan.head()


## 7. Geographic filter

Retain only events within the Europe/Mediterranean bounding box
(lat 0–90 °N, lon −30–60 °E).

In [ ]:
mask = (
    (df_events['ev_latitude']  >= LAT_MIN) & (df_events['ev_latitude']  <= LAT_MAX) &
    (df_events['ev_longitude'] >= LON_MIN) & (df_events['ev_longitude'] <= LON_MAX)
)
n_before = len(df_events)
df_events = df_events[mask].copy()
print(f"Events before filter : {n_before:,}")
print(f"Events after filter  : {len(df_events):,}  ({n_before - len(df_events):,} removed)")


## 8. Figures

### 8.1 Magnitude histograms (linear and log scale)

Bin width = 0.1 magnitude units.

In [ ]:
mag_valid = df_events['magnitude'].dropna()
bins = np.arange(mag_valid.min(), mag_valid.max() + 0.1, 0.1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

for ax, log, suffix in [
    (ax1, False, ''),
    (ax2, True,  ' (Log Scale)'),
]:
    ax.hist(mag_valid, bins=bins, edgecolor='black', color='orange')
    ax.set_title(f'Distribution of Earthquake Magnitudes{suffix}', fontsize=16)
    ax.set_xlabel('Magnitude', fontsize=14)
    ax.set_ylabel('Frequency' + (' (Log Scale)' if log else ''), fontsize=14)
    ax.tick_params(labelsize=12)
    ax.grid(axis='y', alpha=0.75)
    if log:
        ax.set_yscale('log')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'magnitude_histogram.png'),     dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'magnitude_histogram_log.png'), dpi=150, bbox_inches='tight')
plt.show()


### 8.2 Gutenberg–Richter cumulative frequency curve

In [ ]:
unique_mag, counts = np.unique(mag_valid, return_counts=True)
cumulative   = np.cumsum(counts[::-1])[::-1]

plt.figure(figsize=(10, 6))
plt.scatter(unique_mag, cumulative, edgecolor='black', color='steelblue', s=20)
plt.yscale('log')
plt.title('Gutenberg–Richter Cumulative Frequency', fontsize=16)
plt.xlabel('Magnitude', fontsize=14)
plt.ylabel('Cumulative Number of Earthquakes', fontsize=14)
plt.tick_params(labelsize=12)
plt.grid(True, which='both', ls='--')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gutenberg_richter.png'), dpi=150, bbox_inches='tight')
plt.show()


### 8.3 Stacked magnitude histogram by scale

In [ ]:
plt.figure(figsize=(15, 7))
sns.histplot(
    data=df_events, x='magnitude', hue='magnitude_type',
    multiple='stack',
    bins=bins,
    palette='viridis',
)
plt.title('Distribution of Earthquake Magnitudes by Scale', fontsize=16)
plt.xlabel('Magnitude', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.tick_params(labelsize=12)
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'magnitude_histogram_stacked.png'), dpi=150, bbox_inches='tight')
plt.show()


### 8.4 Two-panel summary: stacked histogram + Gutenberg–Richter curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 8))

# Left: stacked histogram
sns.histplot(
    data=df_events, x='magnitude', hue='magnitude_type',
    multiple='stack', bins=bins, palette='viridis', ax=ax1,
)
ax1.set_xlabel('Magnitude', fontsize=14)
ax1.set_ylabel('Frequency', fontsize=14)
ax1.tick_params(labelsize=12)
ax1.grid(axis='y', alpha=0.75)

# Right: Gutenberg–Richter
ax2.scatter(unique_mag, cumulative, edgecolor='black', color='steelblue', s=20)
ax2.set_yscale('log')
ax2.set_xlabel('Magnitude', fontsize=14)
ax2.set_ylabel('Cumulative Number of Earthquakes', fontsize=14)
ax2.tick_params(labelsize=12)
ax2.grid(True, which='both', ls='--')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'magnitude_analysis_subplots.png'), dpi=150, bbox_inches='tight')
plt.show()


### 8.5 Magnitude-scale prevalence bar chart

In [ ]:
# Count events per preferred magnitude scale
type_counts = df_events['magnitude_type'].value_counts().reindex(MAGNITUDE_COLS, fill_value=0)
type_pct    = 100.0 * type_counts / type_counts.sum()

plt.figure(figsize=(12, 6))
type_pct.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Preferred Magnitude Scale — Event Counts (%)', fontsize=16)
plt.xlabel('Magnitude Scale', fontsize=14)
plt.ylabel('Percentage of Events (%)', fontsize=14)
plt.xticks(rotation=0, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'magnitude_type.png'), dpi=150, bbox_inches='tight')
plt.show()


### 8.6 Focal depth histogram

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(df_events['ev_depth_km'].dropna(), bins=50, edgecolor='black', color='orange')
plt.title('Distribution of Focal Depths', fontsize=16)
plt.xlabel('Depth (km)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.tick_params(labelsize=12)
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'depth_histogram.png'), dpi=150, bbox_inches='tight')
plt.show()

print(df_events['ev_depth_km'].describe().to_string())


### 8.7 Magnitude vs. depth (all scale types)

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=df_events, x='ev_depth_km', y='magnitude',
    hue='magnitude_type', alpha=0.5, s=15,
)
plt.title('Magnitude vs. Focal Depth', fontsize=16)
plt.xlabel('Depth (km) — log scale', fontsize=14)
plt.ylabel('Magnitude', fontsize=14)
plt.tick_params(labelsize=12)
plt.xscale('log')
plt.gca().xaxis.set_major_formatter(mticker.ScalarFormatter())
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'magnitude_vs_depth.png'), dpi=150, bbox_inches='tight')
plt.show()


### 8.8 Magnitude vs. depth — per-scale subplots (md and m combined)

In [ ]:
PANEL_TYPES = ['emec_mw', 'mw', 'ml', 'ms', 'mb', ['md', 'm']]
COLORS      = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

n_cols = 3
n_rows = int(np.ceil(len(PANEL_TYPES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.flatten()

for i, mag_type in enumerate(PANEL_TYPES):
    ax = axes[i]
    if isinstance(mag_type, list):
        subset = df_events[df_events['magnitude_type'].isin(mag_type)]
        title  = 'Scale: ' + ', '.join(mag_type)
        sns.scatterplot(
            data=subset, x='ev_depth_km', y='magnitude',
            hue='magnitude_type', ax=ax, alpha=0.5, s=15,
            palette=COLORS[-len(mag_type):],
        )
        ax.legend(title='Scale', fontsize=10)
    else:
        subset = df_events[df_events['magnitude_type'] == mag_type]
        title  = f'Scale: {mag_type}'
        sns.scatterplot(
            data=subset, x='ev_depth_km', y='magnitude',
            ax=ax, alpha=0.5, s=15, color=COLORS[i],
        )
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Depth (km)', fontsize=12)
    ax.set_ylabel('Magnitude', fontsize=12)
    ax.tick_params(labelsize=11)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.grid(True)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, 'magnitude_vs_depth_subplots_combined_colored.png'),
    dpi=150, bbox_inches='tight',
)
plt.show()


### 8.9 Magnitude vs. event time — per-scale subplots (md and m combined)

In [ ]:
df_events['event_time'] = pd.to_datetime(df_events['event_time'])

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.flatten()

for i, mag_type in enumerate(PANEL_TYPES):
    ax = axes[i]
    if isinstance(mag_type, list):
        subset = df_events[df_events['magnitude_type'].isin(mag_type)]
        title  = 'Scale: ' + ', '.join(mag_type)
        sns.scatterplot(
            data=subset, x='event_time', y='magnitude',
            hue='magnitude_type', ax=ax, alpha=0.5, s=15,
            palette=COLORS[-len(mag_type):],
        )
        ax.legend(title='Scale', fontsize=10)
    else:
        subset = df_events[df_events['magnitude_type'] == mag_type]
        title  = f'Scale: {mag_type}'
        sns.scatterplot(
            data=subset, x='event_time', y='magnitude',
            ax=ax, alpha=0.5, s=15, color=COLORS[i],
        )
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Event Time', fontsize=12)
    ax.set_ylabel('Magnitude', fontsize=12)
    ax.tick_params(labelsize=11, axis='x', rotation=30)
    ax.grid(True)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, 'magnitude_vs_time_subplots_combined_colored.png'),
    dpi=150, bbox_inches='tight',
)
plt.show()

# Report the most recent event date per scale
for scale in ['emec_mw', 'ms']:
    latest = df_events[df_events['magnitude_type'] == scale]['event_time'].max()
    print(f"Most recent event with '{scale}': {latest}")


### 8.10 Violin plots of magnitude distribution by scale

In [ ]:
df_melted = df_events.melt(
    id_vars=['esm_event_id'],
    value_vars=MAGNITUDE_COLS,
    var_name='magnitude_type',
    value_name='magnitude_value',
).dropna(subset=['magnitude_value'])

plt.figure(figsize=(12, 6))
sns.violinplot(
    x='magnitude_type', y='magnitude_value',
    hue='magnitude_type', data=df_melted,
    palette='viridis', legend=False,
)
plt.title('Magnitude Distribution by Scale', fontsize=16)
plt.xlabel('Magnitude Scale', fontsize=14)
plt.ylabel('Magnitude Value', fontsize=14)
plt.tick_params(labelsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'magnitude_violinplot.png'), dpi=150, bbox_inches='tight')
plt.show()


## 9. Summary

### Key findings
- The catalogue was deduplicated to one record per `esm_event_id`.
- A preferred magnitude was assigned following the priority order
  `emec_mw → mw → ml → ms → mb → md → m`.
- Events outside the Europe/Mediterranean bounding box were excluded.
- The magnitude distribution follows the expected Gutenberg–Richter relationship,
  with completeness typically above M ≥ 3.

### Possible next steps
- Estimate the **completeness magnitude** (Mc) using maximum-curvature or
  goodness-of-fit methods.
- Fit the Gutenberg–Richter **b-value** by linear regression on the log-linear
  cumulative frequency curve.
- Cross-check `emec_mw` coverage against the original EMEC catalogue to assess
  temporal gaps visible in the time-series plots.
